# LeetCode 1399 — Count Largest Group bằng Python

**Link đề:** https://leetcode.com/problems/count-largest-group/
**Độ khó:** Easy · **Chủ đề:** Hash Table, Math, Counting

```python
class Solution:
    def countLargestGroup(self, n: int) -> int:
        ...
```

## Mục tiêu học
1. Tính **tổng chữ số** (digit sum) bằng `% 10` và `// 10`, và bằng chuỗi.
2. Hiểu ý tưởng **gom nhóm theo khóa** (group by key) bằng `dict` / `Counter` / mảng đếm.
3. Kỹ thuật "**đếm số nhóm có kích thước lớn nhất**": tìm `max` rồi đếm số lần đạt `max`.
4. Mẹo tái sử dụng kết quả: `digit_sum(i) = digit_sum(i // 10) + i % 10` (quy hoạch động nhỏ).
5. Trace, chứng minh, Time/Space, lỗi hay gặp, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ định nghĩa hàm, không tự chạy.

## 1. Đề bài

Cho số nguyên `n`. Gom các số từ `1` tới `n` thành **nhóm** theo **tổng các chữ số** của chúng. Ví dụ `14` (1+4=5) và `5` cùng nhóm; `13` (1+3=4) và `3` khác nhóm.

Trả về **số lượng nhóm có kích thước lớn nhất** (kích thước = số phần tử của nhóm).

| `n` | Output | Giải thích |
|---:|---:|---|
| `13` | `4` | 9 nhóm: `[1,10], [2,11], [3,12], [4,13], [5], [6], [7], [8], [9]` → 4 nhóm có kích thước 2 |
| `2` | `2` | `[1], [2]` đều kích thước 1 |

**Constraints:** `1 <= n <= 10^4`.

Chú ý: đề **không** hỏi kích thước lớn nhất là bao nhiêu, mà hỏi **có bao nhiêu nhóm** đạt kích thước đó.

In [1]:
examples = [(13, 4), (2, 2)]
for n, expected in examples:
    print(f"n = {n} -> kỳ vọng {expected}")

n = 13 -> kỳ vọng 4
n = 2 -> kỳ vọng 2


## 2. Nền tảng Python cần biết

### 2.1. Lấy chữ số cuối và bỏ chữ số cuối: `% 10`, `// 10`

- `x % 10` là **phần dư** khi chia 10 → chữ số hàng đơn vị.
- `x // 10` là **chia lấy phần nguyên** → bỏ chữ số cuối.
- Khác `x / 10` (chia thực, ra `float`).

In [2]:
x = 1234
print("x % 10  =", x % 10)
print("x // 10 =", x // 10)
print("x / 10  =", x / 10)

x % 10  = 4
x // 10 = 123
x / 10  = 123.4


### 2.2. Tính tổng chữ số bằng vòng `while`

Lặp: cộng chữ số cuối vào `total`, rồi bỏ nó đi, cho tới khi `x == 0`.

```text
x = 1234: total 0 -> 4 -> 7 -> 9 -> 10
```

In [3]:
def digit_sum(x: int) -> int:
    total = 0
    while x > 0:
        total += x % 10
        x //= 10
    return total

for x in [5, 14, 13, 1234, 9999, 10000]:
    print(x, "->", digit_sum(x))

5 -> 5
14 -> 5
13 -> 4
1234 -> 10
9999 -> 36
10000 -> 1


### 2.3. Cách bằng chuỗi: `str`, `int`, `sum` + generator

`str(1234)` → `"1234"`; duyệt từng ký tự `ch`, đổi `int(ch)`, rồi `sum(...)`. Dễ đọc, nhưng chậm hơn một chút vì tạo chuỗi.

In [4]:
def digit_sum_str(x: int) -> int:
    return sum(int(ch) for ch in str(x))

print(digit_sum_str(1234), digit_sum_str(9999))
assert all(digit_sum(x) == digit_sum_str(x) for x in range(1, 10001))
print("2 cách tính tổng chữ số khớp nhau trên 1..10000")

10 36
2 cách tính tổng chữ số khớp nhau trên 1..10000


### 2.4. Gom nhóm bằng `dict`: key = tổng chữ số, value = số phần tử

Ta không cần lưu cả danh sách phần tử của nhóm, chỉ cần **kích thước**. `size[key] = size.get(key, 0) + 1`.

Muốn xem nhóm trông thế nào thì dùng `dict` của `list` để học — `setdefault(key, [])` trả về list của key (tạo mới nếu chưa có).

In [5]:
groups = {}
for x in range(1, 14):
    groups.setdefault(digit_sum(x), []).append(x)
for key in sorted(groups):
    print(f"tổng chữ số {key}: {groups[key]}")

tổng chữ số 1: [1, 10]
tổng chữ số 2: [2, 11]
tổng chữ số 3: [3, 12]
tổng chữ số 4: [4, 13]
tổng chữ số 5: [5]
tổng chữ số 6: [6]
tổng chữ số 7: [7]
tổng chữ số 8: [8]
tổng chữ số 9: [9]


### 2.5. `max(d.values())` và `list.count`

- `d.values()` là các value của dict.
- `max(d.values())` = kích thước nhóm lớn nhất.
- `list(d.values()).count(m)` = số nhóm có kích thước `m`.

In [6]:
size = {k: len(v) for k, v in groups.items()}     # dict comprehension
print("size =", size)
m = max(size.values())
print("kích thước lớn nhất =", m)
print("số nhóm đạt max   =", list(size.values()).count(m))

size = {1: 2, 2: 2, 3: 2, 4: 2, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
kích thước lớn nhất = 2
số nhóm đạt max   = 4


`{k: len(v) for k, v in groups.items()}` là **dict comprehension**: tạo dict mới từ từng cặp `(k, v)`.

## 3. Cách trực giác (baseline) — thử từng tổng chữ số, đếm lại từ đầu

Người mới có thể nghĩ: "Tổng chữ số có thể là `1, 2, ..., 36` (vì `9999` có tổng 36). Với mỗi tổng `s`, duyệt lại `1..n` đếm xem bao nhiêu số có tổng `s`."

Cách này đúng nhưng tính `digit_sum` của mỗi số tới 36 lần.

In [7]:
def count_largest_group_bruteforce(n: int) -> int:
    sizes = []
    for s in range(1, 37):                         # mọi tổng chữ số có thể với n <= 10^4
        c = sum(1 for x in range(1, n + 1) if digit_sum(x) == s)
        if c > 0:
            sizes.append(c)
    m = max(sizes)
    return sizes.count(m)

for n, expected in examples:
    print(n, "->", count_largest_group_bruteforce(n), "| kỳ vọng", expected)

13 -> 4 | kỳ vọng 4
2 -> 2 | kỳ vọng 2


### 3.1. Nút thắt

- `36 × n` lần gọi `digit_sum`, mỗi lần `O(số chữ số)`. Với `n = 10^4`: ~360 000 lần gọi.
- Việc **lặp lại**: cùng một số `x` bị tính tổng chữ số 36 lần, dù kết quả không đổi.
- Còn phụ thuộc vào con số "36" — nếu `n` lớn hơn thì phải sửa.

⇒ Đảo ngược vòng lặp: duyệt **mỗi số một lần**, tính tổng chữ số, rồi **bỏ nó vào đúng nhóm**.

In [8]:
calls = 0
def digit_sum_counted(x):
    global calls
    calls += 1
    return digit_sum(x)

n = 10000
for s in range(1, 37):
    sum(1 for x in range(1, n + 1) if digit_sum_counted(x) == s)
print("baseline gọi digit_sum:", calls, "lần; cách gom nhóm chỉ cần", n, "lần")

baseline gọi digit_sum: 360000 lần; cách gom nhóm chỉ cần 10000 lần


`global calls` cho phép hàm **sửa** biến `calls` ở ngoài hàm (chỉ dùng để đo, không dùng trong lời giải).

## 4. Insight — một lượt duyệt, gom nhóm bằng hash map

Mỗi số thuộc **đúng một** nhóm (khóa = tổng chữ số). Vậy:
1. Duyệt `x = 1..n`, tăng `size[digit_sum(x)]`.
2. `m = max(size.values())`.
3. Đếm số khóa có `size == m`.

Đây là pattern **"group by key → thống kê trên kích thước nhóm"**.

## 5. Lời giải chính — giảng từng đoạn

**Đoạn 1:** `size = {}`.

**Đoạn 2:** `for x in range(1, n + 1)` — nhớ `n + 1` vì `range` không lấy điểm cuối.

**Đoạn 3:** `key = digit_sum(x)`; `size[key] = size.get(key, 0) + 1`.

**Đoạn 4:** `m = max(size.values())`; trả `sum(1 for v in size.values() if v == m)`.

In [9]:
def count_largest_group_hashmap(n: int) -> int:
    size = {}                                   # Đoạn 1
    for x in range(1, n + 1):                   # Đoạn 2
        key = digit_sum(x)                      # Đoạn 3
        size[key] = size.get(key, 0) + 1
    m = max(size.values())                      # Đoạn 4
    return sum(1 for v in size.values() if v == m)

for n, expected in examples:
    print(n, "->", count_largest_group_hashmap(n), "| kỳ vọng", expected)

13 -> 4 | kỳ vọng 4
2 -> 2 | kỳ vọng 2


### 5.1. Bản `Counter`

`Counter(digit_sum(x) for x in range(1, n + 1))` gom nhóm trong 1 dòng.

In [10]:
from collections import Counter

def count_largest_group_counter(n: int) -> int:
    cnt = Counter(digit_sum(x) for x in range(1, n + 1))
    m = max(cnt.values())
    return list(cnt.values()).count(m)

print([count_largest_group_counter(n) for n, _ in examples])

[4, 2]


## 6. Cách 3 — mảng đếm + tái sử dụng tổng chữ số (DP nhỏ)

**Quan sát 1:** tổng chữ số tối đa với `n <= 10^4` là `36` (của `9999`) ⇒ dùng list `count` độ dài 37 thay dict.

**Quan sát 2:** `digit_sum(x) = digit_sum(x // 10) + x % 10`. Ví dụ `digit_sum(1234) = digit_sum(123) + 4`. Vì `x // 10 < x`, nếu tính theo thứ tự tăng dần thì `digit_sum(x // 10)` **đã có sẵn** ⇒ mỗi số chỉ tốn `O(1)`.

Đây là **quy hoạch động** (dynamic programming) dạng đơn giản nhất: lưu kết quả bài toán nhỏ để dùng cho bài toán lớn.

In [11]:
n = 25
ds = [0] * (n + 1)                 # ds[x] = tổng chữ số của x, ds[0] = 0
for x in range(1, n + 1):
    ds[x] = ds[x // 10] + x % 10
print(list(enumerate(ds))[10:26])

[(10, 1), (11, 2), (12, 3), (13, 4), (14, 5), (15, 6), (16, 7), (17, 8), (18, 9), (19, 10), (20, 2), (21, 3), (22, 4), (23, 5), (24, 6), (25, 7)]


In [12]:
def count_largest_group_dp(n: int) -> int:
    ds = [0] * (n + 1)
    count = [0] * 37
    for x in range(1, n + 1):
        ds[x] = ds[x // 10] + x % 10
        count[ds[x]] += 1
    m = max(count)
    return count.count(m)

for n, expected in examples:
    print(n, "->", count_largest_group_dp(n), "| kỳ vọng", expected)

13 -> 4 | kỳ vọng 4
2 -> 2 | kỳ vọng 2


**Câu hỏi:** `count[0]` luôn bằng 0 (không số nào từ 1 trở lên có tổng chữ số 0). Có ảnh hưởng `count.count(m)` không? Không, vì `m >= 1` (luôn có ít nhất một nhóm không rỗng), nên các ô `0` không bị đếm.

## 7. Cách 4 — một lượt duyệt, cập nhật max và số nhóm đạt max "trực tuyến"

Không cần duyệt lại `size` ở cuối: mỗi khi một nhóm tăng kích thước lên `c`:
- nếu `c > best_size`: kích thước kỷ lục mới → `best_size = c`, `num_groups = 1`.
- nếu `c == best_size`: thêm một nhóm cùng kỷ lục → `num_groups += 1`.

Đúng vì kích thước mỗi nhóm chỉ **tăng dần từng 1**, nên một nhóm chỉ "chạm" mỗi mức đúng một lần.

In [13]:
def count_largest_group_online(n: int) -> int:
    count = [0] * 37
    best_size, num_groups = 0, 0
    for x in range(1, n + 1):
        key = digit_sum(x)
        count[key] += 1
        c = count[key]
        if c > best_size:
            best_size, num_groups = c, 1
        elif c == best_size:
            num_groups += 1
    return num_groups

for n, expected in examples:
    print(n, "->", count_largest_group_online(n), "| kỳ vọng", expected)

13 -> 4 | kỳ vọng 4
2 -> 2 | kỳ vọng 2


## 8. TRACE từng bước

Trace cách online với `n = 13`:

| x | key = digit_sum | count[key] mới | so với best_size | best_size | num_groups |
|---:|---:|---:|---|---:|---:|
| 1 | 1 | 1 | 1 > 0 → kỷ lục mới | 1 | 1 |
| 2 | 2 | 1 | bằng | 1 | 2 |
| ... | ... | 1 | bằng | 1 | ... |
| 9 | 9 | 1 | bằng | 1 | 9 |
| 10 | 1 | 2 | 2 > 1 → kỷ lục mới | 2 | 1 |
| 11 | 2 | 2 | bằng | 2 | 2 |
| 12 | 3 | 2 | bằng | 2 | 3 |
| 13 | 4 | 2 | bằng | 2 | 4 |

Kết quả `4`. Điểm đáng chú ý: tại `x = 10`, `num_groups` **reset về 1**.

In [14]:
def trace_count_largest_group(n: int) -> int:
    count = {}
    best_size, num_groups = 0, 0
    for x in range(1, n + 1):
        key = digit_sum(x)
        count[key] = count.get(key, 0) + 1
        c = count[key]
        if c > best_size:
            best_size, num_groups, note = c, 1, "kỷ lục mới -> reset"
        elif c == best_size:
            num_groups, note = num_groups + 1, "bằng kỷ lục -> +1"
        else:
            note = "nhỏ hơn -> bỏ qua"
        print(f"x={x:>2} key={key:>2} size={c} | best_size={best_size} num_groups={num_groups} ({note})")
    return num_groups

print("kết quả:", trace_count_largest_group(13))

x= 1 key= 1 size=1 | best_size=1 num_groups=1 (kỷ lục mới -> reset)
x= 2 key= 2 size=1 | best_size=1 num_groups=2 (bằng kỷ lục -> +1)
x= 3 key= 3 size=1 | best_size=1 num_groups=3 (bằng kỷ lục -> +1)
x= 4 key= 4 size=1 | best_size=1 num_groups=4 (bằng kỷ lục -> +1)
x= 5 key= 5 size=1 | best_size=1 num_groups=5 (bằng kỷ lục -> +1)
x= 6 key= 6 size=1 | best_size=1 num_groups=6 (bằng kỷ lục -> +1)
x= 7 key= 7 size=1 | best_size=1 num_groups=7 (bằng kỷ lục -> +1)
x= 8 key= 8 size=1 | best_size=1 num_groups=8 (bằng kỷ lục -> +1)
x= 9 key= 9 size=1 | best_size=1 num_groups=9 (bằng kỷ lục -> +1)
x=10 key= 1 size=2 | best_size=2 num_groups=1 (kỷ lục mới -> reset)
x=11 key= 2 size=2 | best_size=2 num_groups=2 (bằng kỷ lục -> +1)
x=12 key= 3 size=2 | best_size=2 num_groups=3 (bằng kỷ lục -> +1)
x=13 key= 4 size=2 | best_size=2 num_groups=4 (bằng kỷ lục -> +1)
kết quả: 4


Trace bản gom nhóm với `n = 24` để thấy có nhóm **không** đạt max (nhóm 7, 8, 9, 10 chỉ có kích thước 1–2 trong khi kỷ lục là 3):

In [15]:
cnt = Counter(digit_sum(x) for x in range(1, 25))
print("kích thước từng nhóm:", dict(sorted(cnt.items())))
m = max(cnt.values())
print("max =", m, "-> các nhóm đạt max:", [k for k, v in sorted(cnt.items()) if v == m])
print("đáp án:", count_largest_group_online(24))

kích thước từng nhóm: {1: 2, 2: 3, 3: 3, 4: 3, 5: 3, 6: 3, 7: 2, 8: 2, 9: 2, 10: 1}
max = 3 -> các nhóm đạt max: [2, 3, 4, 5, 6]
đáp án: 5


## 9. Code nộp LeetCode

### Bản hash map / Counter

In [16]:
from collections import Counter

class Solution:
    def countLargestGroup(self, n: int) -> int:
        cnt = Counter()
        for x in range(1, n + 1):
            s = 0
            y = x
            while y:
                s += y % 10
                y //= 10
            cnt[s] += 1
        m = max(cnt.values())
        return sum(1 for v in cnt.values() if v == m)

print([Solution().countLargestGroup(n) for n, _ in examples])

[4, 2]


Chú ý dùng biến phụ `y = x` để khỏi phá `x` (vòng `for` vẫn đúng trong Python, nhưng thói quen này tránh lỗi ở ngôn ngữ khác).

### Bản DP + mảng đếm

In [17]:
class SolutionDP:
    def countLargestGroup(self, n: int) -> int:
        ds = [0] * (n + 1)
        count = [0] * 37
        for x in range(1, n + 1):
            ds[x] = ds[x // 10] + x % 10
            count[ds[x]] += 1
        m = max(count)
        return count.count(m)

print([SolutionDP().countLargestGroup(n) for n, _ in examples])

[4, 2]


### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Baseline | thử từng tổng `s`, đếm lại | `O(36 · n · d)` | `O(1)` | lặp việc |
| 2. Hash map / Counter | một lượt, gom theo key | `O(n · d)` | `O(K)` | tổng quát |
| 3. DP + mảng đếm | `ds[x] = ds[x//10] + x%10` | `O(n)` | `O(n)` | nhanh nhất |
| 4. Online max | cập nhật kỷ lục khi đếm | `O(n · d)` | `O(K)` | không duyệt lại |

`d` = số chữ số (≤ 5), `K` = số nhóm (≤ 37).

## 10. Vì sao đúng?

- Mỗi `x ∈ [1, n]` có đúng một tổng chữ số ⇒ thuộc đúng một nhóm ⇒ sau vòng lặp, `size[key]` = kích thước chính xác của nhóm `key` (bất biến: sau khi xử lý `1..x`, `size[key]` = số phần tử trong `1..x` có tổng `key`).
- `m = max(size.values())` là kích thước lớn nhất; đếm các key có `size == m` đúng là số nhóm lớn nhất.
- **Cách online:** bất biến sau mỗi bước: `best_size` = max kích thước hiện tại, `num_groups` = số nhóm đang có kích thước đó. Khi một nhóm tăng lên `c`: nếu `c > best_size` thì chỉ nhóm này đạt mức mới; nếu `c == best_size` thì nhóm này **vừa mới** đạt mức (trước đó là `c - 1`), nên chưa bị đếm ⇒ `+1` không đếm trùng.
- **DP:** `x = 10·(x // 10) + x % 10` ⇒ các chữ số của `x` là các chữ số của `x // 10` cộng thêm chữ số cuối ⇒ công thức đúng; `ds[0] = 0` là trường hợp cơ sở.

## 11. Time / Space

Gọi `d = ⌊log10 n⌋ + 1` (số chữ số, ≤ 5).

- Hash map: `n` lần tính `digit_sum` ⇒ `O(n · d)`; dict có tối đa `9d + 1` key ⇒ `O(d)` bộ nhớ (hằng số nhỏ).
- DP: `O(n)` thời gian, `O(n)` bộ nhớ cho `ds` (có thể bỏ `ds` nếu dùng `digit_sum` trực tiếp).
- Với `n <= 10^4`, mọi cách trừ baseline đều chạy trong vài mili-giây.

In [18]:
import time
for f in [count_largest_group_bruteforce, count_largest_group_hashmap, count_largest_group_dp, count_largest_group_online]:
    t0 = time.perf_counter()
    r = f(10000)
    print(f"{f.__name__:<32} -> {r} ({1000 * (time.perf_counter() - t0):.1f} ms)")

count_largest_group_bruteforce   -> 1 (81.2 ms)
count_largest_group_hashmap      -> 1 (2.4 ms)
count_largest_group_dp           -> 1 (0.8 ms)
count_largest_group_online       -> 1 (3.4 ms)


## 12. Bộ test

Mẫu đề; biên `n = 1` (một nhóm), `n = 9` (9 nhóm kích thước 1), `n = 10` (nhóm 1 có 2 phần tử), `n = 10^4`; và **đối chiếu** mọi cách với baseline cho `n = 1..2000`.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS -> {actual}")

sol = Solution()
check("ví dụ 1", sol.countLargestGroup(13), 4)
check("ví dụ 2", sol.countLargestGroup(2), 2)
check("n = 1", sol.countLargestGroup(1), 1)
check("n = 9", sol.countLargestGroup(9), 9)
check("n = 10", sol.countLargestGroup(10), 1)
check("n = 15", sol.countLargestGroup(15), 6)
check("n = 24", sol.countLargestGroup(24), 5)
check("n = 10000", sol.countLargestGroup(10000), count_largest_group_bruteforce(10000))

ví dụ 1        PASS -> 4
ví dụ 2        PASS -> 2
n = 1          PASS -> 1
n = 9          PASS -> 9
n = 10         PASS -> 1
n = 15         PASS -> 6
n = 24         PASS -> 5


n = 10000      PASS -> 1


In [20]:
methods = [count_largest_group_hashmap, count_largest_group_counter, count_largest_group_dp,
           count_largest_group_online, Solution().countLargestGroup, SolutionDP().countLargestGroup]
for n in range(1, 2001):
    expected = count_largest_group_bruteforce(n) if n <= 300 else count_largest_group_hashmap(n)
    for f in methods:
        assert f(n) == expected, (f, n)
print("Mọi cách khớp nhau trên n = 1..2000")

Mọi cách khớp nhau trên n = 1..2000


## 13. Lỗi người mới hay gặp

### Lỗi 1: Trả **kích thước** lớn nhất thay vì **số nhóm**

In [21]:
def wrong_return_size(n):
    cnt = Counter(digit_sum(x) for x in range(1, n + 1))
    return max(cnt.values())           # SAI: đây là kích thước, không phải số nhóm

print("sai:", wrong_return_size(13), "| đúng:", count_largest_group_hashmap(13))

sai: 2 | đúng: 4


### Lỗi 2: `range(1, n)` bỏ sót `n`

In [22]:
def wrong_range(n):
    cnt = Counter(digit_sum(x) for x in range(1, n))     # SAI: thiếu n
    m = max(cnt.values())
    return list(cnt.values()).count(m)

print("n=10: sai:", wrong_range(10), "| đúng:", count_largest_group_hashmap(10))

n=10: sai: 9 | đúng: 1


### Lỗi 3: Tính tổng chữ số mà làm hỏng biến vòng lặp ngoài

Trong `while` dùng `while`/biến chung với vòng ngoài (ví dụ vòng ngoài cũng dùng `while x <= n` và tự tăng `x`) → `x` bị chia về 0 → vòng lặp vô hạn hoặc sai. Luôn chép sang biến phụ `y = x`.

In [23]:
def wrong_shared_var(n, limit=50):
    cnt = Counter()
    x, steps = 1, 0
    while x <= n and steps < limit:      # thêm limit để demo không treo
        s = 0
        while x:                         # SAI: phá luôn x
            s += x % 10
            x //= 10
        cnt[s] += 1
        x += 1                           # x giờ luôn = 1
        steps += 1
    return steps

print("số vòng chạy (đáng lẽ 13):", wrong_shared_var(13), "-> kẹt vì x bị reset")

số vòng chạy (đáng lẽ 13): 50 -> kẹt vì x bị reset


### Lỗi 4: Mảng đếm quá nhỏ

Nếu tưởng tổng chữ số tối đa là `9×4 = 36` nhưng cấp `[0] * 36` thì `count[36]` lỗi `IndexError` khi `n >= 9999`. Cần `[0] * 37`.

In [24]:
try:
    count = [0] * 36
    count[digit_sum(9999)] += 1
except IndexError as e:
    print("IndexError:", e, "| digit_sum(9999) =", digit_sum(9999))

IndexError: list index out of range | digit_sum(9999) = 36


## 14. Mẹo nhận dạng pattern

- "Gom/nhóm theo **một thuộc tính**" ⇒ dict với key = thuộc tính (`Counter` nếu chỉ cần kích thước).
- "**Số nhóm** có kích thước lớn nhất" ⇒ hai bước `max` rồi `count`, hoặc cập nhật kỷ lục online.
- "Chữ số của số" ⇒ `% 10` / `// 10`; nếu cần cho **mọi** số `1..n` ⇒ DP `ds[x] = ds[x // 10] + x % 10`.
- Khóa có miền nhỏ, biết trước (tổng chữ số ≤ 36) ⇒ mảng đếm thay dict.

## 15. Biến thể

1. Nhóm theo **tích** chữ số, hoặc theo **số chữ số khác 0**.
2. Trả về **danh sách các nhóm lớn nhất** (các khóa).
3. Đếm số nhóm lớn nhất trong đoạn `[lo, hi]` thay vì `[1, n]`.
4. LeetCode 1742 *Maximum Number of Balls in a Box*: hộp = tổng chữ số, trả kích thước hộp lớn nhất (rất giống bài này).

In [25]:
def largest_group_keys(n: int) -> list[int]:
    cnt = Counter(digit_sum(x) for x in range(1, n + 1))
    m = max(cnt.values())
    return sorted(k for k, v in cnt.items() if v == m)

def count_balls(low_limit: int, high_limit: int) -> int:     # LeetCode 1742
    cnt = Counter(digit_sum(x) for x in range(low_limit, high_limit + 1))
    return max(cnt.values())

print("các tổng chữ số của nhóm lớn nhất (n=13):", largest_group_keys(13))
print("1742, [1, 10]:", count_balls(1, 10), "| [5, 15]:", count_balls(5, 15))

các tổng chữ số của nhóm lớn nhất (n=13): [1, 2, 3, 4]
1742, [1, 10]: 2 | [5, 15]: 2


## 16. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Với `n = 20`, liệt kê kích thước các nhóm và cho biết đáp án.

**Bài 2 (đoán output):** `digit_sum(907)`, `907 % 10`, `907 // 10`, `sum(int(c) for c in "907")`.

**Bài 3 (điền chỗ trống):** `exercise_count_largest_group(n)` dùng dict + `max` + đếm.

**Bài 4 (tự viết):** `exercise_digit_sums(n)` trả list `ds` độ dài `n + 1` với `ds[x]` = tổng chữ số của `x`, dùng công thức DP (không dùng vòng `while`).

**Bài 5 (biến thể):** `exercise_largest_product_groups(n)` — gom `1..n` theo **tích** các chữ số, trả số nhóm có kích thước lớn nhất.

**Bài 6 (mini challenge):** `exercise_range_groups(lo, hi)` — như bài chính nhưng cho đoạn `[lo, hi]`, viết **một lượt** với cập nhật kỷ lục online.

In [26]:
def exercise_count_largest_group(n: int) -> int:
    # TODO 1: size = {}
    # TODO 2: duyệt 1..n, tăng size[digit_sum(x)]
    # TODO 3: m = max(...), đếm số value == m
    pass

def exercise_digit_sums(n: int) -> list[int]:
    # TODO: ds[x] = ds[x // 10] + x % 10
    pass

# assert exercise_count_largest_group(13) == 4
# assert exercise_digit_sums(12) == [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 1, 2, 3]

In [27]:
def exercise_largest_product_groups(n: int) -> int:
    # TODO
    pass

def exercise_range_groups(lo: int, hi: int) -> int:
    # TODO: một lượt duyệt, best_size / num_groups
    pass

# assert exercise_largest_product_groups(13) == 3
# assert exercise_range_groups(1, 13) == 4

## 17. Gợi ý 3 mức

**Bài 1** — (1) Liệt kê tổng chữ số của 1..20. (2) Nhóm 1: `1,10`; nhóm 2: `2,11,20`... (3) Đếm nhóm có kích thước lớn nhất.

**Bài 3** — (1) Ba đoạn: gom, max, đếm. (2) `size.get(key, 0) + 1`. (3) `sum(1 for v in size.values() if v == m)`.

**Bài 4** — (1) `ds = [0] * (n + 1)`. (2) Duyệt `x` tăng dần từ 1. (3) `ds[x] = ds[x // 10] + x % 10`.

**Bài 5** — (1) Viết `digit_product(x)` giống `digit_sum` nhưng nhân, khởi tạo `1`. (2) Gom bằng `Counter`. (3) Chú ý số chứa chữ số `0` đều có tích `0` → nhóm 0 thường rất lớn.

**Bài 6** — (1) Duyệt `range(lo, hi + 1)`. (2) Giữ `count`, `best_size`, `num_groups`. (3) `>` thì reset về 1, `==` thì `+1`.

## 18. Đáp án tham khảo

**Bài 1:** tổng chữ số 1..20: nhóm `2` = `{2, 11, 20}` có 3 phần tử; các nhóm `1, 3..9` có 2 phần tử (`1,10`, `3,12`, ..., `9,18`); nhóm `10` = `{19}`. Lớn nhất là 3, chỉ có 1 nhóm ⇒ **1**.

**Bài 2:** `16`, `7`, `90`, `16`.

In [28]:
def answer_count_largest_group(n):
    size = {}
    for x in range(1, n + 1):
        key = digit_sum(x)
        size[key] = size.get(key, 0) + 1
    m = max(size.values())
    return sum(1 for v in size.values() if v == m)

def answer_digit_sums(n):
    ds = [0] * (n + 1)
    for x in range(1, n + 1):
        ds[x] = ds[x // 10] + x % 10
    return ds

def digit_product(x):
    p = 1
    while x:
        p *= x % 10
        x //= 10
    return p

def answer_largest_product_groups(n):
    cnt = Counter(digit_product(x) for x in range(1, n + 1))
    m = max(cnt.values())
    return list(cnt.values()).count(m)

def answer_range_groups(lo, hi):
    count = {}
    best_size = num_groups = 0
    for x in range(lo, hi + 1):
        k = digit_sum(x)
        count[k] = count.get(k, 0) + 1
        if count[k] > best_size:
            best_size, num_groups = count[k], 1
        elif count[k] == best_size:
            num_groups += 1
    return num_groups

assert answer_count_largest_group(20) == 1
assert (digit_sum(907), 907 % 10, 907 // 10, sum(int(c) for c in "907")) == (16, 7, 90, 16)
assert answer_count_largest_group(13) == 4
assert answer_digit_sums(12) == [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 1, 2, 3]
assert answer_largest_product_groups(13) == 3        # nhóm tích 1 = {1, 11}, 2 = {2, 12}, 3 = {3, 13}
assert answer_range_groups(1, 13) == 4
for n in range(1, 500):
    assert answer_range_groups(1, n) == answer_count_largest_group(n) == count_largest_group_dp(n)
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


Giải thích bài 5 với `n = 13`: tích chữ số của `1..9` là chính nó; `10 → 0`, `11 → 1`, `12 → 2`, `13 → 3`. Nhóm tích `1` = `{1, 11}`, `2` = `{2, 12}`, `3` = `{3, 13}` có kích thước 2, các nhóm còn lại kích thước 1 ⇒ đáp án **3**. Với `n` lớn, nhóm tích `0` (mọi số chứa chữ số 0) sẽ áp đảo. In ra để kiểm chứng:

In [29]:
cnt = Counter(digit_product(x) for x in range(1, 14))
print(dict(sorted(cnt.items())))
m = max(cnt.values())
print("max =", m, "| số nhóm đạt max =", list(cnt.values()).count(m))
big = Counter(digit_product(x) for x in range(1, 10001))
print("n = 10^4: nhóm tích 0 có", big[0], "phần tử -> đáp án", answer_largest_product_groups(10000))

{0: 1, 1: 2, 2: 2, 3: 2, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1}
max = 2 | số nhóm đạt max = 3
n = 10^4: nhóm tích 0 có 2620 phần tử -> đáp án 1


## 19. Template nhớ nhanh + checklist submit

```python
from collections import Counter

class Solution:
    def countLargestGroup(self, n: int) -> int:
        cnt = Counter(sum(map(int, str(x))) for x in range(1, n + 1))   # gom theo tổng chữ số
        m = max(cnt.values())                                            # kích thước lớn nhất
        return sum(1 for v in cnt.values() if v == m)                    # số nhóm đạt max
```

`map(int, str(x))` áp `int` lên từng ký tự của chuỗi — cách viết gọn khác của tổng chữ số.

**Khung tư duy "group by key":** chọn key → đếm kích thước nhóm → thống kê (max, số nhóm đạt max, ...).

**Checklist:**
- [ ] Duyệt `range(1, n + 1)` (có `n`).
- [ ] Trả **số nhóm**, không phải kích thước nhóm.
- [ ] Tính tổng chữ số trên biến phụ, không phá biến vòng lặp.
- [ ] Mảng đếm đủ lớn (37 ô cho `n <= 10^4`).